In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import fiona
from pathlib import Path
import matplotlib.pyplot as plt
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
import os
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.windows import Window
from rasterio.plot import show
import scipy.ndimage as nd
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrow
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
import matplotlib
print(matplotlib.rcParams['font.family'])
matplotlib.rcParams['font.family'] = 'Times New Roman'
import sys, pathlib
sys.path.append(str(pathlib.Path("../../../robyns_libraries").resolve()))
import Robyn_paper_2_defs

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
admin_boundaries_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/admin_boundaries.gpkg"

# Optional: see available layers in each GeoPackage
print("Jamaica layers:", fiona.listlayers(jamaica_boundary_path))
print("Admin layers:", fiona.listlayers(admin_boundaries_path))

# Read first layer (or set layer="..." explicitly if needed)
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
admin_boundaries = gpd.read_file(admin_boundaries_path)

print("jamaica_boundary CRS (original):", jamaica_boundary.crs)
print("admin_boundaries CRS (original):", admin_boundaries.crs)

# Reproject both to target CRS
jamaica_boundary = jamaica_boundary.to_crs(jamaica_metric_grid_crs)
admin_boundaries = admin_boundaries.to_crs(jamaica_metric_grid_crs)

# Bounds checks
print("\nJamaica boundary total bounds (minx, miny, maxx, maxy):")
print(jamaica_boundary.total_bounds)

print("\nAdmin boundaries total bounds (minx, miny, maxx, maxy):")
print(admin_boundaries.total_bounds)


In [ ]:
# Path to protected areas shapefile
forest_reserves_path = base_path / "dphil_common_cross_cutting/common_incoming_data/protected_landcover/Forest_reserves.shp"
protected_areas_path = base_path / "dphil_common_cross_cutting/common_incoming_data/protected_landcover/Protected_areas.shp"

# Load + reproject
forest_reserves = gpd.read_file(forest_reserves_path).to_crs(jamaica_metric_grid_crs)
protected_areas = gpd.read_file(protected_areas_path).to_crs(jamaica_metric_grid_crs)

# Combine geometries only
protected_all = gpd.GeoDataFrame(
    pd.concat(
        [forest_reserves[['geometry']], protected_areas[['geometry']]],
        ignore_index=True
    ),
    crs=jamaica_metric_grid_crs
)

# Clean + dissolve to one non-overlapping geometry
protected_all = protected_all[protected_all.geometry.notna()].copy()
protected_all['geometry'] = protected_all.geometry.make_valid()

combined_protected_layers = gpd.GeoDataFrame(
    geometry=[protected_all.geometry.union_all()],
    crs=jamaica_metric_grid_crs
)

In [ ]:
# Paths to coral reef and seagrass shapefiles
coral_reef_path = base_path / "dphil_paper_3/processed_data/corals/corals_clipped_1000m.shp"
seagrass_path = base_path / "dphil_paper_3/processed_data/seagrass/seagrass_clipped_10000m.shp"

# Load the coral reef and seagrass shapefiles
coral_reef = gpd.read_file(coral_reef_path)
seagrass = gpd.read_file(seagrass_path)

# Transform the coral reef and seagrass and admin boundaries layers to the map's CRS
coral_reef = coral_reef.to_crs(jamaica_metric_grid_crs)
seagrass = seagrass.to_crs(jamaica_metric_grid_crs)

In [ ]:
land_use = base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print(terrestrial_landcover.crs)

In [ ]:
categories = terrestrial_landcover['Classify'].unique()
# Display the categories
print("Land Use Categories:")
for category in categories:
    print("-", category)

In [ ]:
# Optional but recommended: remove rows with missing class labels
terrestrial_landcover = terrestrial_landcover.dropna(subset=['Classify']).copy()

terrestrial_landcover['area_m2'] = terrestrial_landcover.geometry.area

area_by_category = (
    terrestrial_landcover
    .groupby('Classify', as_index=False)['area_m2']
    .sum()
    .assign(
        area_km2=lambda d: d['area_m2'] / 1e6,
        percentage=lambda d: (d['area_m2'] / d['area_m2'].sum()) * 100
    )
    .sort_values('percentage', ascending=False)
)

total_area = area_by_category['area_m2'].sum()
print(f"Total Area: {total_area:,.2f} square meters")

# Final summary table
area_summary = (
    area_by_category[['Classify', 'area_km2', 'percentage']]
    .rename(columns={
        'Classify': 'Land Use Category',
        'area_km2': 'Area (km²)',
        'percentage': 'Percentage of Total Area (%)'
    })
    .copy()
)

# Round display values
area_summary['Area (km²)'] = area_summary['Area (km²)'].round(2)
area_summary['Percentage of Total Area (%)'] = area_summary['Percentage of Total Area (%)'].round(2)

# Add total row
totals_row = pd.DataFrame([{
    'Land Use Category': 'Total',
    'Area (km²)': area_by_category['area_km2'].sum(),
    'Percentage of Total Area (%)': 100.0
}])

area_summary = pd.concat([area_summary, totals_row], ignore_index=True)

display(area_summary)

In [ ]:
#Land Use Categories:
#- Agriculture
#- Forest
#- Urban
#- Wetland
#- Grassland

# Example mapping dictionary
landuse_category_mapping = {
    'Bare Rock': 'Bare Rock',
    'Fields: Herbaceous crops, fallow, cultivated vegetables': 'Agriculture',
    'Fields: Pasture,Human disturbed, grassland': 'Agriculture',
    'Herbaceous Wetland': 'Freshwater wetland',
    'Mangrove Forest': 'Mangrove',
    'Fields: Bare Land': 'Agriculture',
    'Open dry forest - Short': 'Open dry forest',
    'Open dry forest - Tall (Woodland/Savanna)': 'Open dry forest',
    'Plantation: Tree crops, shrub crops, sugar cane, banana': 'Plantation',
    'Quarry': 'Bauxite extraction / quarry',
    'Water Body': 'Water body',
    'Buildings and other infrastructures': 'Buildings and other infrastructure',
    'Fields and Secondary Forest': 'Mixed land use: forests with bamboo or agriculture/plantation',
    'Bamboo and Fields': 'Mixed land use: agriculture and bamboo',
    'Bauxite Extraction': 'Bauxite extraction / quarry',
    'Disturbed broadleaved forest (Secondary Forest)': 'Forest',
    'Fields  and Bamboo': 'Mixed land use: agriculture and bamboo',
    'Bamboo and Secondary Forest': 'Mixed land use: agriculture and bamboo',
    'Hardwood Plantation: Euculytus': 'Plantation',
    'Hardwood Plantation: Mixed': 'Plantation',
    'Swamp Forest': 'Swamp forest',
    'Fields or Secondary Forest/Pine Plantation': 'Mixed land use: forests with bamboo or agriculture/plantation',
    'Hardwood Plantation: Mahoe': 'Plantation',
    'Hardwood Plantation: Mahogany': 'Plantation',
    'Bamboo': 'Bamboo',
    'Closed broadleaved forest (Primary Forest)': 'Forest',
    'Secondary Forest': 'Forest'
    # Add other mappings as needed
}

# Map the categories
terrestrial_landcover['Classify'] = terrestrial_landcover['Classify'].replace(landuse_category_mapping)

In [ ]:
# Define custom colors for specified categories

custom_colors = {
    'Bare Rock': '#A9A9A9',  # Dark Gray (rocky terrain)
    'Agriculture': '#8B4513',  # Dark Brown
    'Freshwater wetland': '#4169E1', #Royal blue
    'Mangrove': '#008080', # Teal Blue
    'Open dry forest': '#A4C639', # Light green 90EE90   
    'Plantation': '#F5DEB3', #  yellow
    'Bauxite extraction / quarry': '#B22222', #Iron Oxide Red
    'Water body': '#4682B4',  # Steel Blue
    'Buildings and other infrastructure': '#000000',  # Black
    'Mixed land use: forests with bamboo or agriculture/plantation': '#32CD32',  # lime Green
    'Mixed land use: agriculture and bamboo': '#D2B48C',  #light brown
    'Swamp forest': '#6B8E23',  # Olive Drab
    'Bamboo': '#F4A460',  # Sandy Brown
    'Forest': '#006400',  # Dark Green
}

# Get the list of all categories
all_categories = area_summary['Land Use Category'].tolist()

# Categories without custom colors
remaining_categories = [cat for cat in all_categories if cat not in custom_colors]

# Create a colormap for remaining categories
cmap = plt.colormaps['Set3']  # keep as colormap

# Assign colors to remaining categories
colormap_colors = {}
n = max(1, len(remaining_categories))
for idx, category in enumerate(remaining_categories):
    color = mcolors.rgb2hex(cmap(idx / (n - 1) if n > 1 else 0))
    colormap_colors[category] = color


# Assign colors to remaining categories
colormap_colors = {}
for idx, category in enumerate(remaining_categories):
    color = mcolors.rgb2hex(cmap(idx))
    colormap_colors[category] = color

# Combine custom colors with colormap colors
category_colors = {**custom_colors, **colormap_colors}

# Map colors to the GeoDataFrame
terrestrial_landcover['color'] = terrestrial_landcover['Classify'].map(category_colors)

# Plot the GeoDataFrame with the assigned colors
fig, ax = plt.subplots(figsize=(18, 14), dpi=300) # Larger map size
terrestrial_landcover.plot(
    ax=ax,
    color=terrestrial_landcover['color'],
    linewidth=0.1,
    edgecolor='black'
)

# Plot coral reef after base map, otherwise they might not show up
coral_reef.plot(
    ax=ax,
    color='pink',  # Pink color for coral reefs
    label='Coral Reefs',
    linewidth=1,
    alpha=1, # Optional transparency
    edgecolor='pink'  # Optional edge color
)

# Plot seagrass
seagrass.plot(
    ax=ax,
    color='turquoise',  # Specify a color for seagrass
    label='Seagrass',
    alpha=0.6,  # Optional transparency
    edgecolor='turquoise'  # Optional edge color
)

# Plot administrative boundaries
# admin_boundaries.plot(
#     ax=ax,
#     facecolor='none',  # No fill for administrative boundaries
#     edgecolor='yellow',  # Yellow outline for admin boundaries
#     linewidth=1,  # Line thickness
#     linestyle='--',  # Dashed line style
#     alpha=0.8,  # Optional transparency
#     label='Admin Boundaries'  # Add to legend
# )

# Prepare legend handles
legend_handles = []
for category, percentage in zip(all_categories, area_summary['Percentage of Total Area (%)']):
    color = category_colors[category]
    label = f"{category} ({percentage:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add additional legend entries for coral reef and seagrass
legend_handles.append(mpatches.Patch(color='pink', label='Coral Reefs'))
legend_handles.append(mpatches.Patch(color='turquoise', label='Seagrass'))
# legend_handles.append(
#     Line2D([0], [0], color='yellow', linestyle='--', linewidth=1, label='Administrative Boundaries')
# )


# Add legend below the plot with 3 columns and adjusted spacing
legend = ax.legend(
    handles=legend_handles,
    title='Landcover types and total terrestrial area coverage',
    bbox_to_anchor=(0.5, -0.1),  # Move legend closer to the map
    loc='upper center',
    ncol=3,  # Set to 3 columns
    frameon=False,  # Remove the box around the legend
    fontsize=12,  # Larger font size for legend text
    title_fontsize=14,  # Larger font size for the legend title
    labelspacing=1.0,  # Increase spacing between title and legend items
    prop={'family': 'Times New Roman'}  # Use TNR font for legend text
)

# Adjust the position of the legend title slightly above the legend items
legend.get_title().set_position((0, 10))  # Fine-tune the title position for more space above items


# If you want to use your local function (the one you defined earlier with location=)
Robyn_paper_2_defs.add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)
Robyn_paper_2_defs.add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)

plt.title(
    "Figure 1(a) Baseline assessment of Jamaica's natural terrestrial landcover and marine ecosystems", 
    fontsize=20,  # Larger font size for the title
    fontweight='bold',  # Bold font
    fontname='Times New Roman',  # Use Times New Roman font
    loc='center',  # Center the title
    pad=20  # Add some padding between the title and the map
)

plt.tight_layout()
plt.show()

In [ ]:
# Keep only valid class rows
terrestrial_landcover = terrestrial_landcover.dropna(subset=['Classify']).copy()

# Protected portions of landcover
protected_landuse = gpd.clip(
    terrestrial_landcover[['Classify', 'geometry']].copy(),
    combined_protected_layers
)
protected_landuse['protected_area_m2'] = protected_landuse.geometry.area

# Total area by class (all land)
total_landuse_summary = (
    terrestrial_landcover
    .assign(total_area_m2=terrestrial_landcover.geometry.area)
    .groupby('Classify', as_index=False)['total_area_m2']
    .sum()
)

# Protected area by class
protected_landuse_summary = (
    protected_landuse
    .groupby('Classify', as_index=False)['protected_area_m2']
    .sum()
)

# Join and calculate % protected
landuse_protection = (
    total_landuse_summary
    .merge(protected_landuse_summary, on='Classify', how='left')
    .fillna({'protected_area_m2': 0})
)

landuse_protection['Percentage Protected (%)'] = (
    landuse_protection['protected_area_m2'] / landuse_protection['total_area_m2'] * 100
)

landuse_protection['Total Area (km²)'] = landuse_protection['total_area_m2'] / 1e6
landuse_protection['Protected Area (km²)'] = landuse_protection['protected_area_m2'] / 1e6

landuse_protection = landuse_protection[
    ['Classify', 'Total Area (km²)', 'Protected Area (km²)', 'Percentage Protected (%)']
].sort_values('Percentage Protected (%)', ascending=False)

display(landuse_protection)

# Sanity check
print("Rows >100%:", (landuse_protection['Percentage Protected (%)'] > 100.0001).sum())



In [ ]:
# 1) Hard rule: no class can have protected > total
check = landuse_protection.copy()
check['diff_km2'] = check['Protected Area (km²)'] - check['Total Area (km²)']
bad = check[check['diff_km2'] > 1e-6]
print("Rows with Protected > Total:", len(bad))
display(bad)

# 2) Global consistency check (weighted class % should match direct %)
total_land_m2 = terrestrial_landcover.geometry.area.sum()
protected_land_m2 = gpd.clip(
    terrestrial_landcover[['geometry']].copy(),
    combined_protected_layers
).geometry.area.sum()

direct_pct = protected_land_m2 / total_land_m2 * 100

weighted_pct = (
    (landuse_protection['Percentage Protected (%)'] * landuse_protection['Total Area (km²)']).sum()
    / landuse_protection['Total Area (km²)'].sum()
)

print(f"Direct protected %:   {direct_pct:.6f}")
print(f"Weighted class %:     {weighted_pct:.6f}")
print(f"Abs difference:       {abs(direct_pct - weighted_pct):.10f}")

# 3) Check how much overlap existed in raw protected layers (explains old >100%)
raw_protected = gpd.GeoDataFrame(
    pd.concat([forest_reserves[['geometry']], protected_areas[['geometry']]], ignore_index=True),
    crs=jamaica_metric_grid_crs
)
raw_area = raw_protected.geometry.area.sum()
union_area = gpd.GeoSeries([raw_protected.unary_union], crs=jamaica_metric_grid_crs).area.iloc[0]
print(f"Raw/union area ratio (overlap inflation): {raw_area/union_area:.4f}")


In [ ]:
# combined_protected_layers should already be a dissolved, non-overlapping mask
# Example:
# combined_protected_layers = gpd.GeoDataFrame(
#     geometry=[protected_all.geometry.union_all()],
#     crs=jamaica_metric_grid_crs
# )

def marine_protection_stats(gdf, mask, category_name):
    g = gdf[['geometry']].copy()
    g = g[g.geometry.notna()].copy()
    g['geometry'] = g.geometry.make_valid()

    protected_g = gpd.clip(g, mask)

    total_area_m2 = g.geometry.area.sum()
    protected_area_m2 = protected_g.geometry.area.sum()
    pct = (protected_area_m2 / total_area_m2 * 100) if total_area_m2 > 0 else np.nan

    row = {
        'Category': category_name,
        'Total Area (km²)': total_area_m2 / 1e6,
        'Protected Area (km²)': protected_area_m2 / 1e6,
        'Percentage Protected (%)': pct
    }
    return protected_g, row

# Recompute marine protected areas with clip (no overlap double counting)
protected_coral_reef, coral_row = marine_protection_stats(
    coral_reef.to_crs(jamaica_metric_grid_crs),
    combined_protected_layers,
    'Coral Reefs'
)

protected_seagrass, seagrass_row = marine_protection_stats(
    seagrass.to_crs(jamaica_metric_grid_crs),
    combined_protected_layers,
    'Seagrass'
)

# Terrestrial table from your validated result
terrestrial_summary = landuse_protection.rename(
    columns={'Classify': 'Category'}
)[['Category', 'Total Area (km²)', 'Protected Area (km²)', 'Percentage Protected (%)']]

# Final combined summary
protected_summary = pd.concat(
    [pd.DataFrame([coral_row, seagrass_row]), terrestrial_summary],
    ignore_index=True
)

# Quick checks
print("Marine rows >100%:",
      (protected_summary.loc[protected_summary['Category'].isin(['Coral Reefs', 'Seagrass']),
                             'Percentage Protected (%)'] > 100.0001).sum())

display(protected_summary.sort_values('Percentage Protected (%)', ascending=False))


In [ ]:
# Table: composition of protected terrestrial area by land-use class

# Ensure area column exists
if 'protected_area_m2' not in protected_landuse.columns:
    protected_landuse['protected_area_m2'] = protected_landuse.geometry.area

protected_composition = (
    protected_landuse
    .dropna(subset=['Classify'])
    .groupby('Classify', as_index=False)['protected_area_m2']
    .sum()
)

total_protected_m2 = protected_composition['protected_area_m2'].sum()

protected_composition['Protected Area (km²)'] = protected_composition['protected_area_m2'] / 1e6
protected_composition['Share of Total Protected Area (%)'] = (
    protected_composition['protected_area_m2'] / total_protected_m2 * 100
)

protected_composition = (
    protected_composition[['Classify', 'Protected Area (km²)', 'Share of Total Protected Area (%)']]
    .rename(columns={'Classify': 'Land Use Category'})
    .sort_values('Share of Total Protected Area (%)', ascending=False)
    .reset_index(drop=True)
)

# Optional formatting
protected_composition['Protected Area (km²)'] = protected_composition['Protected Area (km²)'].round(2)
protected_composition['Share of Total Protected Area (%)'] = protected_composition['Share of Total Protected Area (%)'].round(2)

# Optional total row
total_row = pd.DataFrame([{
    'Land Use Category': 'Total',
    'Protected Area (km²)': protected_composition['Protected Area (km²)'].sum(),
    'Share of Total Protected Area (%)': 100.0
}])

protected_composition = pd.concat([protected_composition, total_row], ignore_index=True)

display(protected_composition)


In [ ]:
# --- New mangrove layer stats in protected areas ---
mangroves_fon_path = base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp"

mangroves_fon = gpd.read_file(mangroves_fon_path).to_crs(jamaica_metric_grid_crs)
mangroves_fon = mangroves_fon[mangroves_fon.geometry.notna()].copy()
mangroves_fon["geometry"] = mangroves_fon.geometry.make_valid()

protected_mangroves_fon = gpd.clip(mangroves_fon[["geometry"]].copy(), combined_protected_layers)

fon_total_m2 = mangroves_fon.geometry.area.sum()
fon_protected_m2 = protected_mangroves_fon.geometry.area.sum()
fon_pct = (fon_protected_m2 / fon_total_m2 * 100) if fon_total_m2 > 0 else 0

# --- Existing mangrove result from your current table ---
existing_row = landuse_protection[
    landuse_protection["Classify"].str.strip().str.lower().eq("mangrove")
].copy()

if existing_row.empty:
    raise ValueError("No 'Mangrove' row found in landuse_protection.")

existing_total_km2 = float(existing_row["Total Area (km²)"].iloc[0])
existing_protected_km2 = float(existing_row["Protected Area (km²)"].iloc[0])
existing_pct = float(existing_row["Percentage Protected (%)"].iloc[0])

# --- Side-by-side comparison ---
comparison = pd.DataFrame([
    {
        "Source": "Forces_of_nature mangroves.shp",
        "Total Area (km²)": fon_total_m2 / 1e6,
        "Protected Area (km²)": fon_protected_m2 / 1e6,
        "Percentage Protected (%)": fon_pct
    },
    {
        "Source": "Existing terrestrial landcover Mangrove class",
        "Total Area (km²)": existing_total_km2,
        "Protected Area (km²)": existing_protected_km2,
        "Percentage Protected (%)": existing_pct
    }
])

comparison["Total Area (km²)"] = comparison["Total Area (km²)"].round(2)
comparison["Protected Area (km²)"] = comparison["Protected Area (km²)"].round(2)
comparison["Percentage Protected (%)"] = comparison["Percentage Protected (%)"].round(2)

display(comparison)

# Optional: direct difference
print("Difference in protected % (FoN - Existing):",
      round((fon_pct - existing_pct), 2), "percentage points")


In [ ]:


# Intersect coral reefs with protected areas
protected_coral_reef = gpd.overlay(coral_reef, combined_protected_layers, how='intersection')
protected_coral_reef['area_m2'] = protected_coral_reef.geometry.area
total_coral_area = coral_reef.geometry.area.sum()
protected_coral_area = protected_coral_reef['area_m2'].sum()
coral_reef_percentage_protected = (protected_coral_area / total_coral_area) * 100

# Intersect seagrass with protected areas
protected_seagrass = gpd.overlay(seagrass, combined_protected_layers, how='intersection')
protected_seagrass['area_m2'] = protected_seagrass.geometry.area
total_seagrass_area = seagrass.geometry.area.sum()
protected_seagrass_area = protected_seagrass['area_m2'].sum()
seagrass_percentage_protected = (protected_seagrass_area / total_seagrass_area) * 100

# Combine statistics into a DataFrame
summary_data = [
    {'Category': 'Coral Reefs', 
     'Total Area (km²)': total_coral_area / 1e6, 
     'Protected Area (km²)': protected_coral_area / 1e6, 
     'Percentage Protected (%)': coral_reef_percentage_protected},
    {'Category': 'Seagrass', 
     'Total Area (km²)': total_seagrass_area / 1e6, 
     'Protected Area (km²)': protected_seagrass_area / 1e6, 
     'Percentage Protected (%)': seagrass_percentage_protected}
]

# Include terrestrial land use statistics
for _, row in protected_landuse_summary.iterrows():
    class_name = row['Classify']
    total_area = area_summary.loc[area_summary['Land Use Category'] == class_name, 'Area (km²)'].values[0] * 1e6  # Convert to m²
    protected_area = row['area_m2']
    summary_data.append({
        'Category': class_name,
        'Total Area (km²)': total_area / 1e6,
        'Protected Area (km²)': protected_area / 1e6,
        'Percentage Protected (%)': (protected_area / total_area) * 100
    })

protected_summary = pd.DataFrame(summary_data)

# Display the summary
print(protected_summary)

# # Calculate total area of protected land use
# total_protected_area = protected_area_by_category['area_m2'].sum()

# # Calculate percentage for each category
# protected_area_by_category['percentage'] = (protected_area_by_category['area_m2'] / total_protected_area) * 100

# # Convert area to square kilometers
# protected_area_by_category['area_km2'] = protected_area_by_category['area_m2'] / 1e6

# # Round the percentage
# protected_area_by_category['percentage'] = protected_area_by_category['percentage'].round(2)

# # Select and reorder columns
# protected_area_summary = protected_area_by_category[['Classify', 'area_km2', 'percentage']]

# # Rename columns for clarity
# protected_area_summary.columns = ['Land Use Category', 'Area (km²)', 'Percentage of Total Protected Area (%)']

# # Display the summary table
# display(protected_area_summary)

# Map colors to the protected land use GeoDataFrame
protected_landuse['color'] = protected_landuse['Classify'].map(category_colors)

# Plot the protected land use map with additional layers
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot the protected land use with assigned colors
protected_landuse.plot(
    ax=ax,
    color=protected_landuse['color'],
    linewidth=0.1,
    edgecolor='black',
    label='Protected Land Use'
)

# Plot protected coral reefs
protected_coral_reef.plot(
    ax=ax,
    color='pink',
    edgecolor='pink',
    alpha=0.8,
    linewidth=1,
    label='Protected Coral Reefs'
)

# Plot protected seagrass
protected_seagrass.plot(
    ax=ax,
    color='turquoise',
    edgecolor='turquoise',
    alpha=0.6,
    linewidth=1,
    label='Protected Seagrass'
)


# Add Jamaica boundaries
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    alpha=0.8
)

# # Prepare legend handles
# legend_handles = []
# for category, percentage in zip(
#     protected_area_summary['Land Use Category'], 
#     protected_area_summary['Percentage of Total Protected Area (%)']
# ):
#     color = category_colors[category]
#     label = f"{category} ({percentage:.2f}%)"
#     patch = mpatches.Patch(color=color, label=label)
#     legend_handles.append(patch)

# # Add legend entry for protected coral reefs
# legend_handles.append(
#     mpatches.Patch(color='pink', label='Coral Reefs')
# )

# # Add legend entry for protected seagrass
# legend_handles.append(
#     mpatches.Patch(color='turquoise', label='Seagrass')
# )

# # Add additional legend entries for admin boundaries
# legend_handles.append(
#     Line2D([0], [0], color='black', linestyle='--', linewidth=1, label='Jamaica terrestrial boundary')
# )



# Prepare legend handles
legend_handles = []

# Percentages for protected terrestrial classes only
total_protected_landuse = protected_landuse_summary['area_m2'].sum()

for _, row in protected_landuse_summary.iterrows():
    category = row['Classify']
    percentage = (row['area_m2'] / total_protected_landuse) * 100
    color = category_colors.get(category, '#cccccc')
    label = f"{category} ({percentage:.2f}%)"
    legend_handles.append(mpatches.Patch(color=color, label=label))

# Add legend entry for protected coral reefs
legend_handles.append(mpatches.Patch(color='pink', label='Coral Reefs'))

# Add legend entry for protected seagrass
legend_handles.append(mpatches.Patch(color='turquoise', label='Seagrass'))

# Add Jamaica boundary legend entry
legend_handles.append(
    Line2D([0], [0], color='black', linestyle='--', linewidth=1, label='Jamaica terrestrial boundary')
)


# Add legend below the plot
legend = ax.legend(
    handles=legend_handles,
    title='Protected Land Cover Types and Area Coverage',
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

# Adjust the position of the legend title
legend.get_title().set_position((0, 10))

# Add the north arrow
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)

# Add the scale bar
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Title for the map
plt.title(
    "Figure 1(b) Baseline assessment of protected natural terrestrial and marine ecosystems", 
    fontsize=20, 
    fontweight='bold', 
    fontname='Times New Roman', 
    loc='center', 
    pad=20
)

plt.tight_layout()
plt.show()


# Optional: round for readability
protected_summary_export = protected_summary.copy()
protected_summary_export['Total Area (km²)'] = protected_summary_export['Total Area (km²)'].round(2)
protected_summary_export['Protected Area (km²)'] = protected_summary_export['Protected Area (km²)'].round(2)
protected_summary_export['Percentage Protected (%)'] = protected_summary_export['Percentage Protected (%)'].round(2)

# Export full table (includes coral + seagrass + terrestrial classes)
out_csv = base_path / "dphil_paper_3/processed_data/protected_summary.csv"
protected_summary_export.to_csv(out_csv, index=False)

# Optional: terrestrial land-use only (exclude marine rows)
landuse_only = protected_summary_export[
    ~protected_summary_export['Category'].isin(['Coral Reefs', 'Seagrass'])
]
out_csv_landuse = base_path / "dphil_paper_3/processed_data/protected_landuse_summary.csv"
landuse_only.to_csv(out_csv_landuse, index=False)

# Optional Excel
out_xlsx = base_path / "dphil_paper_3/results/protected_summary.xlsx"
protected_summary_export.to_excel(out_xlsx, index=False)


In [ ]:
# Define custom colors for specified categories

custom_colors = {
    'Bare Rock': '#A9A9A9',  # Dark Gray (rocky terrain)
    'Agriculture': '#8B4513',  # Dark Brown
    'Freshwater wetland': '#4169E1', #Royal blue
    'Mangrove': '#008080', # Teal Blue
    'Open dry forest': '#A4C639', # Light green 90EE90   
    'Plantation': '#F5DEB3', #  yellow
    'Bauxite extraction / quarry': '#B22222', #Iron Oxide Red
    'Water body': '#4682B4',  # Steel Blue
    'Buildings and other infrastructure': '#000000',  # Black
    'Mixed land use: forests with bamboo or agriculture/plantation': '#32CD32',  # lime Green
    'Mixed land use: agriculture and bamboo': '#D2B48C',  #light brown
    'Swamp forest': '#6B8E23',  # Olive Drab
    'Bamboo': '#F4A460',  # Sandy Brown
    'Forest': '#006400',  # Dark Green
}

# Get the list of all categories
all_categories = area_summary['Land Use Category'].tolist()

# Categories without custom colors
remaining_categories = [cat for cat in all_categories if cat not in custom_colors]

# Create a colormap for remaining categories
#cmap = plt.cm.get_cmap('Set3', len(remaining_categories))
cmap = plt.colormaps['Set3'](len(remaining_categories))

# Assign colors to remaining categories
colormap_colors = {}
for idx, category in enumerate(remaining_categories):
    color = mcolors.rgb2hex(cmap(idx))
    colormap_colors[category] = color

# Combine custom colors with colormap colors
category_colors = {**custom_colors, **colormap_colors}

# Map colors to the GeoDataFrame
terrestrial_landcover['color'] = terrestrial_landcover['Classify'].map(category_colors)


# Plot the GeoDataFrame with the assigned colors
fig, ax = plt.subplots(figsize=(18, 14), dpi=300) # Larger map size
terrestrial_landcover.plot(
    ax=ax,
    color=terrestrial_landcover['color'],
    linewidth=0.1,
    edgecolor='black'
)


# Plot coral reef after base map, otherwise they might not show up
coral_reef.plot(
    ax=ax,
    color='pink',  # Pink color for coral reefs
    label='Coral Reefs',
    linewidth=1,
    alpha=1, # Optional transparency
    edgecolor='pink'  # Optional edge color
)

# Plot seagrass
seagrass.plot(
    ax=ax,
    color='turquoise',  # Specify a color for seagrass
    label='Seagrass',
    alpha=0.6,  # Optional transparency
    edgecolor='turquoise'  # Optional edge color
)

# Plot administrative boundaries
# admin_boundaries.plot(
#     ax=ax,
#     facecolor='none',  # No fill for administrative boundaries
#     edgecolor='yellow',  # Yellow outline for admin boundaries
#     linewidth=1,  # Line thickness
#     linestyle='--',  # Dashed line style
#     alpha=0.8,  # Optional transparency
#     label='Admin Boundaries'  # Add to legend
# )

# Prepare legend handles
legend_handles = []
for category, percentage in zip(all_categories, area_summary['Percentage of Total Area (%)']):
    color = category_colors[category]
    label = f"{category} ({percentage:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add additional legend entries for coral reef and seagrass
legend_handles.append(mpatches.Patch(color='pink', label='Coral Reefs'))
legend_handles.append(mpatches.Patch(color='turquoise', label='Seagrass'))
# legend_handles.append(
#     Line2D([0], [0], color='yellow', linestyle='--', linewidth=1, label='Administrative Boundaries')
# )


# Add legend below the plot with 3 columns and adjusted spacing
legend = ax.legend(
    handles=legend_handles,
    title='Landcover types and total terrestrial area coverage',
    bbox_to_anchor=(0.5, -0.1),  # Move legend closer to the map
    loc='upper center',
    ncol=3,  # Set to 3 columns
    frameon=False,  # Remove the box around the legend
    fontsize=12,  # Larger font size for legend text
    title_fontsize=14,  # Larger font size for the legend title
    labelspacing=1.0,  # Increase spacing between title and legend items
    prop={'family': 'Times New Roman'}  # Use TNR font for legend text
)

# Adjust the position of the legend title slightly above the legend items
legend.get_title().set_position((0, 10))  # Fine-tune the title position for more space above items
    
# Add the scale bar closer to the north arrow
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Add the north arrow
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)

plt.title(
    "Figure 1(a) Baseline assessment of Jamaica's natural terrestrial landcover and marine ecosystems", 
    fontsize=20,  # Larger font size for the title
    fontweight='bold',  # Bold font
    fontname='Times New Roman',  # Use Times New Roman font
    loc='center',  # Center the title
    pad=20  # Add some padding between the title and the map
)

plt.tight_layout()
plt.show()


# Recalculate area in square meters for the intersected geometries
protected_landuse['area_m2'] = protected_landuse.geometry.area

# Group by 'Classify' and sum the areas
protected_landuse_summary = protected_landuse.groupby('Classify')['area_m2'].sum().reset_index()

# Intersect coral reefs with protected areas
protected_coral_reef = gpd.overlay(coral_reef, combined_protected_layers, how='intersection')
protected_coral_reef['area_m2'] = protected_coral_reef.geometry.area
total_coral_area = coral_reef.geometry.area.sum()
protected_coral_area = protected_coral_reef['area_m2'].sum()
coral_reef_percentage_protected = (protected_coral_area / total_coral_area) * 100

# Intersect seagrass with protected areas
protected_seagrass = gpd.overlay(seagrass, combined_protected_layers, how='intersection')
protected_seagrass['area_m2'] = protected_seagrass.geometry.area
total_seagrass_area = seagrass.geometry.area.sum()
protected_seagrass_area = protected_seagrass['area_m2'].sum()
seagrass_percentage_protected = (protected_seagrass_area / total_seagrass_area) * 100

# Combine statistics into a DataFrame
summary_data = [
    {'Category': 'Coral Reefs', 
     'Total Area (km²)': total_coral_area / 1e6, 
     'Protected Area (km²)': protected_coral_area / 1e6, 
     'Percentage Protected (%)': coral_reef_percentage_protected},
    {'Category': 'Seagrass', 
     'Total Area (km²)': total_seagrass_area / 1e6, 
     'Protected Area (km²)': protected_seagrass_area / 1e6, 
     'Percentage Protected (%)': seagrass_percentage_protected}
]

# Include terrestrial land use statistics
for _, row in protected_landuse_summary.iterrows():
    class_name = row['Classify']
    total_area = area_summary.loc[area_summary['Land Use Category'] == class_name, 'Area (km²)'].values[0] * 1e6  # Convert to m²
    protected_area = row['area_m2']
    summary_data.append({
        'Category': class_name,
        'Total Area (km²)': total_area / 1e6,
        'Protected Area (km²)': protected_area / 1e6,
        'Percentage Protected (%)': (protected_area / total_area) * 100
    })

protected_summary = pd.DataFrame(summary_data)

# Display the summary
#print(protected_summary)

# # Calculate total area of protected land use
# total_protected_area = protected_area_by_category['area_m2'].sum()

# # Calculate percentage for each category
# protected_area_by_category['percentage'] = (protected_area_by_category['area_m2'] / total_protected_area) * 100

# # Convert area to square kilometers
# protected_area_by_category['area_km2'] = protected_area_by_category['area_m2'] / 1e6

# # Round the percentage
# protected_area_by_category['percentage'] = protected_area_by_category['percentage'].round(2)

# # Select and reorder columns
# protected_area_summary = protected_area_by_category[['Classify', 'area_km2', 'percentage']]

# # Rename columns for clarity
# protected_area_summary.columns = ['Land Use Category', 'Area (km²)', 'Percentage of Total Protected Area (%)']

# # Display the summary table
# display(protected_area_summary)

# Map colors to the protected land use GeoDataFrame
protected_landuse['color'] = protected_landuse['Classify'].map(category_colors)

# Plot the protected land use map with additional layers
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot the protected land use with assigned colors
protected_landuse.plot(
    ax=ax,
    color=protected_landuse['color'],
    linewidth=0.1,
    edgecolor='black',
    label='Protected Land Use'
)

# Plot protected coral reefs
protected_coral_reef.plot(
    ax=ax,
    color='pink',
    edgecolor='pink',
    alpha=0.8,
    linewidth=1,
    label='Protected Coral Reefs'
)

# Plot protected seagrass
protected_seagrass.plot(
    ax=ax,
    color='turquoise',
    edgecolor='turquoise',
    alpha=0.6,
    linewidth=1,
    label='Protected Seagrass'
)


# Add Jamaica boundaries
jamaica_boundary.plot(
    ax=ax,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    alpha=0.8
)

# Prepare legend handles
legend_handles = []
for category, percentage in zip(
    protected_area_summary['Land Use Category'], 
    protected_area_summary['Percentage of Total Protected Area (%)']
):
    color = category_colors[category]
    label = f"{category} ({percentage:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add legend entry for protected coral reefs
legend_handles.append(
    mpatches.Patch(color='pink', label='Coral Reefs')
)

# Add legend entry for protected seagrass
legend_handles.append(
    mpatches.Patch(color='turquoise', label='Seagrass')
)

# Add additional legend entries for admin boundaries
legend_handles.append(
    Line2D([0], [0], color='black', linestyle='--', linewidth=1, label='Jamaica terrestrial boundary')
)

# Add legend below the plot
legend = ax.legend(
    handles=legend_handles,
    title='Protected ecosystem types and their proportion protected',
    bbox_to_anchor=(0.5, -0.1),
    loc='upper center',
    ncol=3,
    frameon=False,
    fontsize=12,
    title_fontsize=14,
    labelspacing=1.0,
    prop={'family': 'Times New Roman'}
)

# Adjust the position of the legend title
legend.get_title().set_position((0, 10))

# Add the north arrow
add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03)

# Add the scale bar
add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, label_offset=0.04, km_offset=0.01)

# Title for the map
plt.title(
    "Figure 1(b) Baseline assessment of protected natural terrestrial and marine ecosystems", 
    fontsize=20, 
    fontweight='bold', 
    fontname='Times New Roman', 
    loc='center', 
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Create a figure with two subplots stacked vertically
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 28), dpi=300, gridspec_kw={'height_ratios': [1, 1]})

# ======= Figure 1(a): Baseline Land Cover Map =======
# Plot the baseline map of land cover
terrestrial_landcover.plot(
    ax=ax1,
    color=terrestrial_landcover['color'],
    linewidth=0.1,
    edgecolor='black'
)

# Plot coral reefs
coral_reef.plot(
    ax=ax1,
    color='pink',
    edgecolor='pink',
    alpha=0.8,
    linewidth=1
)

# Plot seagrass
seagrass.plot(
    ax=ax1,
    color='turquoise',
    edgecolor='turquoise',
    alpha=0.6,
    linewidth=1
)

# Add Jamaica boundary
jamaica_boundary.plot(
    ax=ax1,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    alpha=0.8
)

# Add title for Figure 1(a)
ax1.set_title(
    "Figure 1(a) Baseline assessment of Jamaica's natural terrestrial landcover and marine ecosystems",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    pad=20
)

# ======= Figure 1(b): Protected Areas Map =======
# Plot the protected land use map
protected_landuse.plot(
    ax=ax2,
    color=protected_landuse['color'],
    linewidth=0.1,
    edgecolor='black'
)

# Plot protected coral reefs
protected_coral_reef.plot(
    ax=ax2,
    color='pink',
    edgecolor='pink',
    alpha=0.8,
    linewidth=1
)

# Plot protected seagrass
protected_seagrass.plot(
    ax=ax2,
    color='turquoise',
    edgecolor='turquoise',
    alpha=0.6,
    linewidth=1
)

# Add Jamaica boundary
jamaica_boundary.plot(
    ax=ax2,
    facecolor='none',
    edgecolor='black',
    linewidth=1,
    linestyle='--',
    alpha=0.8
)

# Add title for Figure 1(b)
ax2.set_title(
    "Figure 1(b) Baseline assessment of protected natural terrestrial and marine ecosystems",
    fontsize=20,
    fontweight='bold',
    fontname='Times New Roman',
    pad=20
)

# ======= Legends for Both Maps =======
# Combine legend handles for both maps
legend_handles = []

# Add terrestrial land cover categories
for category, percentage in zip(
    area_summary['Land Use Category'], 
    area_summary['Percentage of Total Area (%)']
):
    color = category_colors[category]
    label = f"{category} ({percentage:.2f}%)"
    patch = mpatches.Patch(color=color, label=label)
    legend_handles.append(patch)

# Add coral reefs and seagrass
legend_handles.append(mpatches.Patch(color='pink', label='Coral Reefs'))
legend_handles.append(mpatches.Patch(color='turquoise', label='Seagrass'))

# Add Jamaica boundary
legend_handles.append(
    Line2D([0], [0], color='black', linestyle='--', linewidth=1, label='Jamaica Boundary')
)

# Add legend below both plots
fig.legend(
    handles=legend_handles,
    title="Land Cover Types and Protected Area Coverage",
    loc="lower center",
    ncol=4,
    fontsize=12,
    title_fontsize=14,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01)
)

# Add tight layout
plt.tight_layout(rect=[0, 0.05, 1, 1])  # Adjust layout to make space for the legend
plt.savefig("Figure_1_combined.png", dpi=300, bbox_inches="tight")
plt.show()